# Cross-Platform Fairness Evaluation (CPFE) — synthetic demonstration

Runs the five-axis CPFE protocol on a **synthetic** two-platform fixture: a high-signal reference platform and a platform with an injected over-confident-but-wrong shift. The numbers illustrate the protocol's behaviour; they are not real-world measurements.

In [ ]:
import numpy as np

from fairscope.nlp import CPFEProtocol, jaccard_topk

rng = np.random.default_rng(20260627)

In [ ]:
def probs(y, strength, nc=4):
    lg = rng.normal(0, 1, (len(y), nc))
    lg[np.arange(len(y)), y] += strength
    e = np.exp(lg - lg.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)

def shifted(y, nc=4):
    fake = (y + rng.integers(1, nc, len(y))) % nc
    lg = rng.normal(0, 1, (len(y), nc))
    lg[np.arange(len(y)), fake] += 3.0
    e = np.exp(lg - lg.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)

yk = rng.integers(0, 4, 1500)
yt = rng.integers(0, 4, 1200)
data = {
    'kaggle': {'y_true': yk, 'probs': probs(yk, 3.0)},
    'twitter': {'y_true': yt, 'probs': shifted(yt)},
}

In [ ]:
report = CPFEProtocol(data, reference='kaggle', n_classes=4, n_boot=300).run()
report.to_dataframe()

**Axis 1 (discrimination)** and **Axis 2 (calibration)**: macro-AUC drops and ECE rises on the shifted platform.

In [ ]:
report.significance['twitter']  # Axis 3: bootstrap macro-AUC test + Bonferroni

In [ ]:
report.equity['twitter']['disparate_impact']  # Axis 4: per-class disparate impact

In [ ]:
# Axis 5 (attribution stability): Jaccard overlap of top-K saliency tokens
sa = {'sad': 0.9, 'tired': 0.8, 'hopeless': 0.7, 'down': 0.6, 'empty': 0.5}
sb = {'market': 0.9, 'price': 0.8, 'stock': 0.7, 'hopeless': 0.6, 'rally': 0.5}
jaccard_topk(sa, sb, 5)

In [ ]:
report.deployment_readiness()  # structured per-axis diagnostic, NOT a decision

`deployment_readiness()` is a **diagnostic**, not a compliance verdict. Equity uses the four-fifths rule and calibration uses stated reference bands; the discrimination $\Delta$AUC limit is illustrative and user-configurable. The five axes are independent — a shift can fail discrimination and calibration while equity passes.